In [3]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression,Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

from sklearn.decomposition import PCA

just to simplfy we removed some columns, their existences is not making much difference to the models performance

In [4]:
df = pd.read_csv('/Users/meetsudra/Documents/GitHub/real-estate/above csvs/gurgaon_properties_post_feature_selection_2.csv').drop(columns=['store room','floor_category','balcony','pooja room','others','study room'])
df.head()

,property_type,sector,price,built_up_area,bedRoom,bathroom,agePossession,servant room,furnishing_type,luxury_category
0,flat,sector 86,0.78,1360.0,2,2,New Property,0,1,Medium
1,flat,sector 103,1.05,1365.0,2,2,Relatively New,0,2,Medium
2,flat,sector 70a,0.97,1339.0,3,3,Relatively New,0,1,Medium
3,flat,sector 69,2.05,1889.0,4,4,Relatively New,0,1,High
4,flat,sector 92,1.83,2104.0,4,3,New Property,1,1,Medium


The goal of this module is to find relationship between X and y. We want to find konse feature price ko kis tarah se affect karte he 

This module will let user understand that what factors will either increase or decrease price and by what percentage so that user can make an educated decision.

So basically will do Inference using linear regression model

In [5]:
# 1 --> unfurnished
# 2 --> semi
# 0 --> furnished

# Numerical = bedRoom, bathroom, built_up_area, servant room
# Ordinal = property_type, sector, agePossession, luxury_category
# OHE = sector, agePossession

In [9]:
# making 3 categories out of 5 to make it easier, altho while developing model with 5 or 3 categories it was revealed that there was not much difference in the r2 scores of both the models 
df['agePossession'] = df['agePossession'].replace(
    {
    'Relatively New' : 'New',
    'Moderately Old':'Old',
    'New Property' : 'New',
    'Old Property' : 'Old',
    'Under Construction' : 'Under Construction'
    }
)


In [10]:
df.head()

,property_type,sector,price,built_up_area,bedRoom,bathroom,agePossession,servant room,furnishing_type,luxury_category
0,flat,sector 86,0.78,1360.0,2,2,New,0,1,Medium
1,flat,sector 103,1.05,1365.0,2,2,New,0,2,Medium
2,flat,sector 70a,0.97,1339.0,3,3,New,0,1,Medium
3,flat,sector 69,2.05,1889.0,4,4,New,0,1,High
4,flat,sector 92,1.83,2104.0,4,3,New,1,1,Medium


In [11]:
df['property_type'] = df['property_type'].replace({'flat':0, 'house':1})

/var/folders/8h/xtp_v7md6bs9195xxzbbf5d80000gn/T/ipykernel_1979/3269943006.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['property_type'] = df['property_type'].replace({'flat':0, 'house':1})


In [12]:
df.head()

,property_type,sector,price,built_up_area,bedRoom,bathroom,agePossession,servant room,furnishing_type,luxury_category
0,0,sector 86,0.78,1360.0,2,2,New,0,1,Medium
1,0,sector 103,1.05,1365.0,2,2,New,0,2,Medium
2,0,sector 70a,0.97,1339.0,3,3,New,0,1,Medium
3,0,sector 69,2.05,1889.0,4,4,New,0,1,High
4,0,sector 92,1.83,2104.0,4,3,New,1,1,Medium


In [13]:
df['luxury_category'] = df['luxury_category'].replace({'High':2, 'Medium':1, 'Low':0})

/var/folders/8h/xtp_v7md6bs9195xxzbbf5d80000gn/T/ipykernel_1979/668286068.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['luxury_category'] = df['luxury_category'].replace({'High':2, 'Medium':1, 'Low':0})


In [14]:
df.head()

,property_type,sector,price,built_up_area,bedRoom,bathroom,agePossession,servant room,furnishing_type,luxury_category
0,0,sector 86,0.78,1360.0,2,2,New,0,1,1
1,0,sector 103,1.05,1365.0,2,2,New,0,2,1
2,0,sector 70a,0.97,1339.0,3,3,New,0,1,1
3,0,sector 69,2.05,1889.0,4,4,New,0,1,2
4,0,sector 92,1.83,2104.0,4,3,New,1,1,1


In [21]:
new_df = pd.get_dummies(df, columns=['sector', 'agePossession'], drop_first=True, dtype=int)

In [22]:
X = new_df.drop(columns=['price'])
y = new_df['price']

In [23]:
y_log = np.log1p(y)

In [24]:
y_log

0       0.576613
1       0.717840
2       0.678034
3       1.115142
4       1.040277
          ...   
3673    1.000632
3674    0.688135
3675    0.788457
3676    1.702928
3677    0.727549
Name: price, Length: 3678, dtype: float64

In [27]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

In [28]:
pd.DataFrame(X_scaled,columns=X.columns)

,property_type,built_up_area,bedRoom,bathroom,servant room,furnishing_type,luxury_category,sector_gwal pahari,sector_manesar,sector_sector 1,...,sector_sector 92,sector_sector 93,sector_sector 95,sector_sector 99,sector_sector 99a,sector_sector 9a,sector_sohna road,sector_sohna road road,agePossession_Old,agePossession_Under Construction
0,-0.50611,-0.408513,-0.871748,-0.868136,-0.740697,-0.421886,0.450791,-0.070129,-0.092196,-0.040423,...,-0.170584,-0.05477,-0.127683,-0.0661,-0.093684,-0.05477,-0.213957,-0.05477,-0.60143,-0.296385
1,-0.50611,-0.404379,-0.871748,-0.868136,-0.740697,1.472736,0.450791,-0.070129,-0.092196,-0.040423,...,-0.170584,-0.05477,-0.127683,-0.0661,-0.093684,-0.05477,-0.213957,-0.05477,-0.60143,-0.296385
2,-0.50611,-0.425878,-0.063304,-0.174608,-0.740697,-0.421886,0.450791,-0.070129,-0.092196,-0.040423,...,-0.170584,-0.05477,-0.127683,-0.0661,-0.093684,-0.05477,-0.213957,-0.05477,-0.60143,-0.296385
3,-0.50611,0.028905,0.745140,0.518921,-0.740697,-0.421886,1.883814,-0.070129,-0.092196,-0.040423,...,-0.170584,-0.05477,-0.127683,-0.0661,-0.093684,-0.05477,-0.213957,-0.05477,-0.60143,-0.296385
4,-0.50611,0.206683,0.745140,-0.174608,1.350080,-0.421886,0.450791,-0.070129,-0.092196,-0.040423,...,5.862200,-0.05477,-0.127683,-0.0661,-0.093684,-0.05477,-0.213957,-0.05477,-0.60143,-0.296385
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3673,-0.50611,-0.210063,-0.063304,-0.174608,1.350080,1.472736,1.883814,-0.070129,-0.092196,-0.040423,...,-0.170584,-0.05477,-0.127683,-0.0661,-0.093684,-0.05477,-0.213957,-0.05477,-0.60143,-0.296385
3674,-0.50611,-0.160450,-0.063304,0.518921,1.350080,-0.421886,-0.982233,-0.070129,-0.092196,-0.040423,...,-0.170584,-0.05477,-0.127683,-0.0661,-0.093684,-0.05477,-0.213957,-0.05477,-0.60143,-0.296385
3675,-0.50611,-0.204275,-0.063304,-0.868136,-0.740697,1.472736,0.450791,-0.070129,-0.092196,-0.040423,...,-0.170584,-0.05477,-0.127683,-0.0661,-0.093684,-0.05477,-0.213957,-0.05477,-0.60143,-0.296385
3676,-0.50611,1.071597,0.745140,0.518921,-0.740697,1.472736,0.450791,-0.070129,-0.092196,-0.040423,...,-0.170584,-0.05477,-0.127683,-0.0661,-0.093684,-0.05477,-0.213957,-0.05477,-0.60143,-0.296385


In [29]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(LinearRegression(),X_scaled,y_log,cv=kfold,scoring='r2')

In [31]:
scores.mean(), scores.std()

(0.8509716091451042, 0.02433400807654884)

In [32]:
lr = LinearRegression()

In [33]:
lr.fit(X_scaled,y_log)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [35]:
lr.coef_.shape

(121,)

using these coefficients we will understand which feature is affecting how to model

In [42]:
pd.DataFrame(lr.coef_.reshape(1,121),columns=X.columns).stack().reset_index().drop(columns=['level_0']).rename(columns={'level_1':'feature',0:'coef'})

,feature,coef
0,property_type,0.119383
1,built_up_area,0.210049
2,bedRoom,0.054372
3,bathroom,0.065531
4,servant room,0.049441
...,...,...
116,sector_sector 9a,-0.005411
117,sector_sohna road,-0.027913
118,sector_sohna road road,-0.009395
119,agePossession_Old,-0.008145


# Regression Analysis

In [44]:
# importing lib
import statsmodels.api as sm

X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)
# add a constant to X
X_with_const = sm.add_constant(X_scaled_df)

# Fit the model
model = sm.OLS(y_log, X_with_const).fit()

# obtain summary stats 
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                  price   R-squared:                       0.866
Model:                            OLS   Adj. R-squared:                  0.861
Method:                 Least Squares   F-statistic:                     189.7
Date:                Mon, 21 Jul 2025   Prob (F-statistic):               0.00
Time:                        15:35:50   Log-Likelihood:                 640.25
No. Observations:                3678   AIC:                            -1036.
Df Residuals:                    3556   BIC:                            -278.9
Df Model:                         121                                         
Covariance Type:            nonrobust                                         
                                        coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
const 

Users asks : 
if i increase bedroom from 1 to 2 then price kitna badhega, assuming everything is same?

- x₂ --- y = 0.5 × x₂  (coef_ = 0.5)
* 1  --- 0.5
* 2  --- 1.0
* 3  --- 1.5

But in our process we applied y_log and x_scaled, we directly cant apply above interpretation. These transformation cant be ignored

when we use scaling, recieved coefficients are called standardized Coefficients
To make them unStandardized coefficients

Stan_coef = Unstan_coef * std(X)/std(log(y)) {Here X is any column from the dataframe}

In [45]:
y_log.std()

0.5552365394042419

In [48]:
# std(X) unscaled X 
X['bedRoom'].std()

1.2371118133242973

In [51]:
# Stan_coef - bedRoom (0.0544)
0.0544 * (0.555/1.237)

# this could be Unstan_coef, but y_log needs to be reversed  

0.024407437348423603

In [54]:
# this is the real coefficient of bedRoom column
Unstan_coef = np.expm1(0.024407437348423603)
Unstan_coef

# If the price of house with 1 bedRoom is 1 cr then house with 2 bedRoom will be 1.024cr

0.02470773705206254

In [ ]:
# lets do it for built_up_area 

built_up_area_coef = 0.2100 
Unstan_coef = np.expm1((built_up_area_coef * y_log.std())/X['built_up_area'].std())
Unstan_coef


9.640515437065593e-05

In [57]:
Unstan_coef*100

0.009640515437065593